# Forecast-period ensemble scalar metrics

Global (cosine-latitude weighted) ensemble verification metrics and rank histograms for the atmospheric hindcasts (`fc`), tabulated and plotted per case. Maps of the same metrics are in `03_compute_and_plot_map_metrics.ipynb`; both notebooks derive from the same verification step, so the global values are consistent with the maps.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states. Model fields are the monthly means over both forecast months, verified against monthly ERA5, GPCP and CERES-OAFlux.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
workflow_name = "fcst_atm_model_evaluation"

component = "atm"
run_segment = "fc"
# Forecast cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"),  # 201201-201202
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # 201206-201207
}
variables = ("TS", "TREFHT", "PSL", "TMQ", "PRECT", "FLUT")
plot_metrics = (
    "bias_global", "rmse_global", "spread_global", "crps_global",
    "ssr", "coverage_global", "pattern_correlation",
)
include_rank_histogram = True
n_bootstrap = 0  # bootstrap CIs are map-only; kept so caches match the map notebook settings

# Model input averaged over the forecast months. None -> monthly h0 means
# ("clim") for forecasts; "daily" or "6hourly" applies to every variable.
model_source = None
lat_range = (-90, 90)
lon_range = (-180, 180)
max_members = None  # e.g. 2 for a quick test; caches record the ensemble size
random_seed = 42
# Caches are rebuilt automatically when they are missing, older than any
# model/reference input, or were built with other settings; force_compute
# rebuilds them regardless.
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.model_evaluation import SCALAR_METRICS, evaluate_cases, experiment_colors, plot_rank_histograms, plot_scalar_metrics, scalar_summary

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
unknown = set(plot_metrics) - set(SCALAR_METRICS)
if unknown:
    raise ValueError(f"Unknown scalar metrics {sorted(unknown)}; choose from {SCALAR_METRICS}")

## 3. Compute or reuse scalar metrics

In [ ]:
results, registries, failures = evaluate_cases(
    "scalar", cases, variables, data_root=INPUT_DATA_ROOT, cache_dir=data_dir,
    component=component, run_segment=run_segment, model_source=model_source,
    n_bootstrap=n_bootstrap, random_seed=random_seed,
    include_rank_histogram=include_rank_histogram,
    max_members=max_members, lat_range=lat_range, lon_range=lon_range,
    force_compute=force_compute,
)
if failures:
    print(f"\n{len(failures)} case(s) failed and are skipped below:")
    for season, variable, experiment, error in failures:
        print(f"  {season} {variable} {experiment}: {error}")

## 4. Summary table

In [ ]:
summary = scalar_summary(results, registries)
table_path = data_dir / f"scalar_metrics_summary_{run_segment}.csv"
summary.to_csv(table_path, index=False)
print(f"[TABLE] {table_path}")
summary

## 5. Plot scalar metrics

In [ ]:
# One figure per case; experiments keep the same color in every figure.
colors = experiment_colors(summary["experiment"])
for season, table in summary.groupby("season", sort=False):
    period = table["period"].iloc[0]
    fig = plot_scalar_metrics(
        table, plot_metrics, variables=variables, colors=colors,
        title=f"Global scalar metrics ({season} {run_segment}, {period})",
    )
    output = figure_dir / f"scalar_metrics_{season}_{run_segment}.png"
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {output}")
    if not show:
        plt.close(fig)

## 6. Rank histograms

In [ ]:
# A flat histogram indicates a well-dispersed ensemble; a U shape indicates under-dispersion.
if include_rank_histogram:
    for season, by_variable in results.items():
        registry = registries[season]
        for variable, datasets in by_variable.items():
            period = registry[next(iter(datasets))]["period"]
            fig = plot_rank_histograms(
                datasets, variable, colors=colors,
                label_map={name: registry[name]["label"] for name in datasets},
                title=f"{variable}: rank histograms ({season} {run_segment}, {period})",
            )
            output = figure_dir / f"{variable}_rank_histogram_{season}_{run_segment}.png"
            fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
            print(f"[FIGURE] {output}")
            if not show:
                plt.close(fig)

## Notes: scalar metrics

| Metric | Definition |
|---|---|
| `model_global_mean`, `reference_global_mean` | global mean of the time-mean ensemble mean / reference |
| `model_rms`, `reference_rms` | spatial standard deviation of those climatologies |
| `rmse_global` | spatial RMSE of the ensemble-mean climatology against the reference |
| `pattern_correlation` | spatial correlation of the climatology anomalies |
| `bias_global`, `spread_global`, `crps_global`, `coverage_global` | global means of the corresponding maps |
| `ssr` | spread-skill ratio, `spread_global / rmse_global` (1 = well dispersed) |
| `bias_squared`, `spread_squared`, `mse`, `sem` | error-decomposition terms; `sem` = spread / sqrt(members) |
| `rank_histogram` | frequency of the reference's rank among the members over all valid samples |

Atmosphere variables: `FLUT` (CERES-OAFlux), `PRECT` (GPCP), `TMQ`, `TS`, `TREFHT`, and `PSL` (ERA5).

Caches: `<experiment>_<variable>_<period>_scalar_metrics.nc`; table: `scalar_metrics_summary_<run_segment>.csv`.